In [1]:
# Dependencies
import pandas as pd
import numpy as np

# Smart Recurrence Detection Algorithm
def smart_detect_recurrence(df):
    df["is_recurring"] = False
    df["recurrence_type"] = None

    for (user, merchant), group in df.groupby(["userId", "merchantName"]):
        if len(group) < 3:
            continue

        group = group.sort_values("date")
        amounts = group["amount"].values
        dates = pd.to_datetime(group["date"]).sort_values().tolist()

        mean_amount = np.mean(amounts)
        std_amount = np.std(amounts)
        if mean_amount == 0 or (std_amount / mean_amount) > 0.10:
            continue

        intervals = [(dates[i + 1] - dates[i]).days for i in range(len(dates) - 1)]
        avg_interval = np.mean(intervals) if intervals else None

        if avg_interval is None or not (25 <= avg_interval <= 35):
            continue

        mask = (df["userId"] == user) & (df["merchantName"] == merchant)
        df.loc[mask, "is_recurring"] = True
        first_type = df.loc[mask].iloc[0]["type"]
        df.loc[mask, "recurrence_type"] = "income" if first_type == "credit" else "expense"

    print(f"Smart recurrence detection completed. Found {df['is_recurring'].sum()} recurring transactions.")
    return df

# Recurrence Detection Detailed Evaluation
def evaluate_recurrence_detailed(df, category_column='subcategory'):
    detailed_categories = {
        'Strongly Expected Recurring': {
            'Rent / Housing': 0,
            'Subscription Services': 0,
            'Salary': 0,
            'Utilities': 0,
            'Insurance': 0
        },
        'Semi-Expected Recurring': {
            'Savings / Transfers to Savings': 0,
            'Freelance': 0,
            'Remittance': 0
        },
        'Typically Non-Recurring': {
            'Peer-to-peer Transfer': 0,
            'Public Transport': 0,
            'Pharmacy / Health': 0,
            'Entertainment & Leisure': 0,
            'Flights / Relocation': 0
        },
        'Mostly Non-Recurring': {
            'Clothing': 0,
            'E-commerce': 0,
            'Beauty & Wellness': 0,
            'Gambling': 0,
            'Groceries': 0
        }
    }

    print("\nDetailed Recurrence Detection Report")
    print("-------------------------------------")

    for group_name, subcats in detailed_categories.items():
        print(f"\n{group_name}:")
        for subcat in subcats:
            mask = df[category_column].str.lower() == subcat.lower()
            total = df[mask].shape[0]
            recurring = df[mask & (df['is_recurring'])].shape[0]
            rate = (recurring / total * 100) if total > 0 else 0
            detailed_categories[group_name][subcat] = rate
            print(f"  {subcat}: {rate:.0f}% flagged as recurring")

    return detailed_categories




In [ ]:
def run_recurrence_pipeline():
    df = pd.read_csv('400_users_transactions.csv')
    print(f"Loaded dataset with {len(df)} transactions.")

    print("\nStarting Recurrence Detection Pipeline...")
    df = smart_detect_recurrence(df)
    report = evaluate_recurrence_detailed(df)

    output_file = '400_users_transactions_with_recurrence.csv'
    df.to_csv(output_file, index=False)
    print(f"\nPipeline execution completed. Updated dataset saved as '{output_file}'.")
    return df, report
